<a href="https://colab.research.google.com/github/petarmimica/Arp299sims/blob/master/EuroMM_stats.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ucitavanje podataka

In [1]:
!curl -o data.csv "https://raw.githubusercontent.com/petarmimica/EuroMM-stats/main/eurojackpot.csv?token=GHSAT0AAAAAAB5IPEQUIHRO7DTYYQTAJG4GY7FJCCA"

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 18037  100 18037    0     0   115k      0 --:--:-- --:--:-- --:--:--  115k


In [2]:
import pandas as pd
data = pd.read_csv('data.csv')
data.head()

,date,prim1,prim2,prim3,prim4,prim5,sec1,sec2
0,2023-02-07,10,15,25,37,46,7,12
1,2023-02-03,1,7,17,44,50,2,10
2,2023-01-31,20,21,30,41,43,10,11
3,2023-01-27,4,9,29,34,37,3,12
4,2023-01-24,9,16,17,27,31,1,9


# Godine i kola

In [23]:
print(f'Ukupno kola: {data.shape[0]}')

Ukupno kola: 614


In [24]:
god_data = data.copy()
god_data['ds'] = pd.to_datetime(god_data['date'])
god_data['godina'] = god_data['ds'].dt.year
print('Broj kola u svakoj godini:')
god_data.groupby('godina')['godina'].count()

Broj kola u svakoj godini:


godina
2012    41
2013    52
2014    52
2015    52
2016    53
2017    52
2018    52
2019    52
2020    52
2021    53
2022    92
2023    11
Name: godina, dtype: int64

# Provjera ponovljenih petica

In [3]:
import numpy as np
data['code5'] = data.apply(lambda row: 
                           (row['prim1'] - 1) * 50**4 + (row['prim2'] - 1) * 50**3 + (row['prim3'] - 1) * 50**2 + 
                           (row['prim4'] - 1) * 50 + row['prim5'] - 1, axis=1)
data['scode5'] = data.apply(lambda row: '_'.join([str(row[x]) for x in ['prim1', 'prim2', 'prim3', 'prim4', 'prim5']]), axis=1)

In [4]:
all_5 = data['code5'].values.tolist()
un_5 = data['code5'].unique().tolist()
print(f'Broj izaslih petica: {len(all_5)}, broj jedinstvenih petica: {len(list(set(un_5)))}')

Broj izaslih petica: 614, broj jedinstvenih petica: 614


In [5]:
all_5 = data['scode5'].values.tolist()
un_5 = data['scode5'].unique().tolist()
print(f'(alt.) Broj izaslih petica: {len(all_5)}, broj jedinstvenih petica: {len(list(set(un_5)))}')

(alt.) Broj izaslih petica: 614, broj jedinstvenih petica: 614


**Nema ponovljenih petica u 609 kola!**

In [6]:
# Provjera ponovljenih cetvorki
four_data = []
for cols in [['prim1', 'prim2', 'prim3', 'prim4'], ['prim1', 'prim2', 'prim3', 'prim5'], ['prim1', 'prim3', 'prim4', 'prim5'], ['prim1', 'prim2', 'prim4', 'prim5'],
             ['prim2', 'prim3', 'prim4', 'prim5']]:
  tmp_data = data.copy()
  tmp_data['code4'] = tmp_data.apply(lambda row: sum([row[cols[i]] * 50 ** (3 - i) for i in range(4)]), axis=1)
  tmp_data['scode4'] = tmp_data.apply(lambda row: '_'.join([str(row[cols[i]]) for i in range(4)]), axis=1)
  four_data.append(tmp_data)
four_data = pd.concat(four_data).sort_values(by='date', ascending=False).reset_index()

In [7]:
all_4 = four_data['code4'].values.tolist()
un_4 = four_data['code4'].unique().tolist()
print(f'Broj izaslih cetvorki: {len(all_4)}, broj jedinstvenih cetvorki: {len(list(set(un_4)))}')

Broj izaslih cetvorki: 3070, broj jedinstvenih cetvorki: 3054


In [8]:
all_4 = four_data['scode4'].values.tolist()
un_4 = four_data['scode4'].unique().tolist()
print(f'(alt.) Broj izaslih cetvorki: {len(all_4)}, broj jedinstvenih cetvorki: {len(list(set(un_4)))}')

(alt.) Broj izaslih cetvorki: 3070, broj jedinstvenih cetvorki: 3054


**Postoji 16 ponovljenih cetvorki**!

In [9]:
tmp = four_data.groupby('code4')['code4'].count()
dups = tmp[tmp > 1].index.tolist()
four_data[four_data['code4'].isin(dups)].sort_values(by=['code4', 'date']).drop(columns=['index', 'code5', 'code4'])

,date,prim1,prim2,prim3,prim4,prim5,sec1,sec2,scode5,scode4
1557,2018-01-05,2,7,38,40,45,7,10,2_7_38_40_45,2_7_38_45
1427,2018-07-06,2,7,24,38,45,5,8,2_7_24_38_45,2_7_38_45
1606,2017-10-27,2,20,23,29,50,4,5,2_20_23_29_50,2_23_29_50
324,2022-06-28,2,10,23,29,50,3,10,2_10_23_29_50,2_23_29_50
1239,2019-03-29,4,9,15,24,42,8,9,4_9_15_24_42,4_9_15_24
882,2020-08-07,4,9,15,24,28,2,6,4_9_15_24_28,4_9_15_24
1946,2016-07-08,4,19,21,31,42,5,10,4_19_21_31_42,4_21_31_42
1319,2018-12-07,4,16,21,31,42,4,6,4_16_21_31_42,4_21_31_42
1491,2018-04-06,6,8,16,23,50,4,8,6_8_16_23_50,6_8_16_50
526,2021-12-17,6,8,16,44,50,1,10,6_8_16_44_50,6_8_16_50


# Par-nepar

In [12]:
pn_data = data.copy()
pn_data['parnih'] = ((pn_data[['prim1', 'prim2', 'prim3', 'prim4', 'prim5']].values - 1) % 2).astype(int).sum(axis=1)
pn_data['neparnih'] = 5 - pn_data['parnih']

In [38]:
print('Frekvencija kola sa određenim brojem parnih brojeva:')
pn_data['parnih'].value_counts()

Frekvencija kola sa određenim brojem parnih brojeva:


2    197
3    185
1    110
4     88
5     20
0     14
Name: parnih, dtype: int64

In [40]:
print('Postotak kola sa određenim brojem parnih brojeva:')
(pn_data['parnih'].value_counts(normalize=True) * 100).round(1)

Postotak kola sa određenim brojem parnih brojeva:


2    32.1
3    30.1
1    17.9
4    14.3
5     3.3
0     2.3
Name: parnih, dtype: float64